In [0]:
# Importing necessary libraries
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import  SparkSession
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("Production_ETL") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [0]:
# Func to upload and upsert
def upsert_to_schema(df, target_schema, target_table, join_key):
    full_table_path = f"{target_schema}.{target_table}"
    
    if not spark.catalog.tableExists(full_table_path):
        print(f"🚀 Table {full_table_path} does not exist, creating...")
        df.write.format("delta").mode("overwrite").saveAsTable(full_table_path)
    else:
        print(f"🔄 Table {full_table_path} exists, performing merge...")
        target_delta_table = DeltaTable.forName(spark, full_table_path)
        (target_delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"✅ Upsert complete for {target_table}")
    

In [0]:
# Loading the delta tables 
instagram_df = spark.read.table("instagram.bronzelayer.instagram_usage_lifestyle")
upsert_to_schema(instagram_df,"instagram","realbronzelayer.instagram_usage_lifestyle","user_id")
